# Phase 5 — Evaluasi akhir dan perbandingan
Angka dibaca dari hasil prediksi model tersimpan pada seluruh test set grouped_v1. Prioritas pemeringkatan adalah Macro F1. Precision/Recall tabel utama memakai macro average.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'PRD.md').exists():
    ROOT = ROOT.parent
if not (ROOT / 'PRD.md').exists():
    raise RuntimeError('Buka notebook dari root proyek atau direktori notebooks')
sys.path.insert(0, str(ROOT))
from src.audit import describe_dataset


In [2]:
metrics = pd.read_csv(ROOT / 'reports/grouped_v1/metrics.csv')
print(metrics.to_string(index=False))
per_class = pd.read_csv(ROOT / 'reports/grouped_v1/per_class_metrics.csv')
print(per_class.to_string(index=False))


              model  accuracy  precision_macro  recall_macro  f1_macro  precision_weighted  recall_weighted  f1_weighted
logistic_regression  0.815326         0.690930      0.726790  0.698483            0.850171         0.815326     0.829413
         linear_svm  0.838135         0.685048      0.678215  0.681168            0.832560         0.838135     0.835225
        naive_bayes  0.821128         0.650765      0.593447  0.572594            0.783195         0.821128     0.790022
              model    class  precision   recall  f1-score  support
        naive_bayes Negative   0.817238 0.886364  0.850398   5060.0
        naive_bayes  Neutral   0.307692 0.005326  0.010471    751.0
        naive_bayes Positive   0.827364 0.888650  0.856912   4185.0
logistic_regression Negative   0.891300 0.803755  0.845267   5060.0
logistic_regression  Neutral   0.278452 0.488682  0.354761    751.0
logistic_regression Positive   0.903038 0.887933  0.895422   4185.0
         linear_svm Negative   0.863876 

In [3]:
result = json.loads((ROOT / 'reports/grouped_v1/evaluation.json').read_text(encoding='utf-8'))
for name, info in result['models'].items():
    print(name, 'confusion matrix (actual rows, predicted columns):', result['classes'])
    print(info['confusion_matrix'])
print('Majority baseline:', result['majority_baseline'])
print('Zero-vector test rows:', result['zero_vector_test_rows'])


naive_bayes confusion matrix (actual rows, predicted columns): ['Negative', 'Neutral', 'Positive']
[[4485, 3, 572], [543, 4, 204], [460, 6, 3719]]
logistic_regression confusion matrix (actual rows, predicted columns): ['Negative', 'Neutral', 'Positive']
[[4067, 703, 290], [275, 367, 109], [221, 248, 3716]]
linear_svm confusion matrix (actual rows, predicted columns): ['Negative', 'Neutral', 'Positive']
[[4417, 336, 307], [396, 197, 158], [300, 121, 3764]]
Majority baseline: {'class_selected_from_train': 'Negative', 'metrics': {'accuracy': 0.506202480992397, 'precision_macro': 0.168734160330799, 'recall_macro': 0.3333333333333333, 'f1_macro': 0.2240524264966348, 'precision_weighted': 0.256240951762858, 'recall_weighted': 0.506202480992397, 'f1_weighted': 0.34024768249488957}}
Zero-vector test rows: 26


In [4]:
verification = json.loads((ROOT / 'reports/grouped_v1/evaluation_verification.json').read_text(encoding='utf-8'))
print(json.dumps({key: verification[key] for key in ['test_rows', 'models', 'feature_manifest_unchanged']}, indent=2))


{
  "test_rows": 9996,
  "models": {
    "naive_bayes": {
      "reload_test_predictions_identical": true,
      "metrics_recomputed_match": true,
      "confusion_matrix_covers_all_test_rows": true
    },
    "logistic_regression": {
      "reload_test_predictions_identical": true,
      "metrics_recomputed_match": true,
      "confusion_matrix_covers_all_test_rows": true
    },
    "linear_svm": {
      "reload_test_predictions_identical": true,
      "metrics_recomputed_match": true,
      "confusion_matrix_covers_all_test_rows": true
    }
  },
  "feature_manifest_unchanged": true
}


## Grafik dari prediksi aktual

![Perbandingan model](../reports/grouped_v1/figures/model_comparison.png)

![Confusion matrix](../reports/grouped_v1/figures/confusion_matrices.png)

![F1 per kelas](../reports/grouped_v1/figures/per_class_f1.png)


## Keterbatasan
Evaluasi menggunakan satu held-out test. Label berasal dari rating, dan subset deduplikasi serta cleaning lama tetap dipakai. Vektor kosong tidak dihapus dari evaluasi. Test yang telah dievaluasi tidak digunakan untuk tuning berikutnya. Peringkat tidak menyatakan signifikansi statistik.
